# Estratégia 3 — SVM e Árvore de Decisão / Random Forest

## 1. Objetivo

Nesta etapa foram implementados três modelos supervisionados adicionais para a tarefa de classificação binária Ham/Spam: SVM (Support Vector Machine), Árvore de Decisão e Random Forest. Assim como na Estratégia 2 (Naive Bayes e Regressão Logística), o objetivo é avaliar o desempenho desses modelos utilizando as representações Bag-of-Words (BoW) e TF-IDF geradas no pré-processamento.

Além do treinamento "padrão", esta etapa testa oversampling (RandomOverSampler) como estratégia adicional de tratamento do desbalanceamento (≈83% Spam / 17% Ham), comparando os resultados com a versão sem balanceamento, de forma equivalente ao que foi feito anteriormente com `class_weight='balanced'` para Regressão Logística e `class_prior` para Naive Bayes.

A validação é feita com Stratified K-Fold (k=5), usando a mesma configuração (`random_state=42`) adotada anteriormente, garantindo que os resultados das duas estratégias sejam diretamente comparáveis.

## 2. Instalação de dependências e Carregamento do Dataset

In [ ]:
# Biblioteca necessária para o RandomOverSampler
!pip install -q imbalanced-learn

In [ ]:
import pandas as pd
import pickle
from google.colab import files

print("Selecione os arquivos: X_bow.pkl, X_tfidf.pkl e labels.pkl")
uploaded = files.upload()

with open('X_bow.pkl', 'rb') as f:
    X_bow = pickle.load(f)

with open('X_tfidf.pkl', 'rb') as f:
    X_tfidf = pickle.load(f)

with open('labels.pkl', 'rb') as f:
    y = pickle.load(f)

print("Dados carregados com sucesso!")
print(f"Dimensão Bag-of-Words: {X_bow.shape}")
print(f"Dimensão TF-IDF: {X_tfidf.shape}")
print(f"Distribuição de classes:\n{y.value_counts()}")

Selecione os arquivos: X_bow.pkl, X_tfidf.pkl e labels.pkl


Saving labels.pkl to labels.pkl
Saving X_bow.pkl to X_bow.pkl
Saving X_tfidf.pkl to X_tfidf.pkl
Dados carregados com sucesso!
Dimensão Bag-of-Words: (450, 3000)
Dimensão TF-IDF: (450, 3000)
Distribuição de classes:
classe_binaria
Spam    375
Ham      75
Name: count, dtype: int64


## 3. Configuração da Validação Cruzada Estratificada

Para manter a consistência com a Estratégia 2, usamos exatamente a mesma configuração de Stratified K-Fold (k=5, `shuffle=True`, `random_state=42`). Isso garante que a proporção original (≈83% Spam / 17% Ham) seja preservada em todas as dobras e que os splits sejam comparáveis entre as duas estratégias.

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 4. Definição dos Modelos e Estratégias de Balanceamento

Foram definidos três classificadores:

- **SVM** (`SVC` com `kernel='linear'`), com busca de hiperparâmetro sobre `C`;
- **Árvore de Decisão** (`DecisionTreeClassifier`), com busca sobre `max_depth` e `min_samples_leaf`;
- **Random Forest** (`RandomForestClassifier`), com busca sobre `n_estimators` e `max_depth`.

Cada classificador é treinado sobre as duas representações (BoW e TF-IDF), em duas versões:

- **Padrão**: treinamento direto, sem tratamento de desbalanceamento;
- **Balanceado**: usando **oversampling** da classe minoritária (Ham) via `RandomOverSampler` (biblioteca `imbalanced-learn`).

A métrica de otimização do `GridSearchCV` é o F1-Score macro, por ser mais sensível ao desempenho da classe minoritária (Ham) do que a acurácia.

In [ ]:
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler

# Grades de hiperparâmetros a testar (
param_grid_svm = {'C': [0.01, 0.1, 1.0, 10.0]}
param_grid_dt = {'max_depth': [5, 10, 20, None], 'min_samples_leaf': [1, 3, 5]}
param_grid_rf = {'n_estimators': [100, 300], 'max_depth': [10, 20, None]}

def get_param_grid(nome_modelo, prefixo=''):
    # Retorna a grade de hiperparâmetros correta para o modelo, aplicando o
    # prefixo 'model__' quando o estimador é um Pipeline (versão balanceada).
    if 'SVM' in nome_modelo:
        base = param_grid_svm
    elif 'RandomForest' in nome_modelo:
        base = param_grid_rf
    elif 'ArvoreDecisao' in nome_modelo:
        base = param_grid_dt
    else:
        raise ValueError(f"Modelo não reconhecido: {nome_modelo}")
    return {f'{prefixo}{k}': v for k, v in base.items()}

def construir_classificador(nome_clf):
    if nome_clf == 'SVM':
        return SVC(kernel='linear', random_state=42)
    if nome_clf == 'ArvoreDecisao':
        return DecisionTreeClassifier(random_state=42)
    if nome_clf == 'RandomForest':
        return RandomForestClassifier(random_state=42, n_jobs=-1)
    raise ValueError(f"Classificador não reconhecido: {nome_clf}")

representacoes = {'BoW': X_bow, 'TFIDF': X_tfidf}
classificadores = ['SVM', 'ArvoreDecisao', 'RandomForest']

# Monta o dicionário de modelos: 3 classificadores x 2 representações x 2 versões
modelos_base = {}
for nome_clf in classificadores:
    for nome_rep, X in representacoes.items():

        # Versão padrão (sem tratamento de desbalanceamento)
        nome_padrao = f'{nome_clf}_{nome_rep}_Padrao'
        modelos_base[nome_padrao] = {
            'estimator': construir_classificador(nome_clf),
            'X': X,
            'balanceado': False,
        }

        # Versão balanceada (oversampling aplicado só no treino de cada dobra)
        nome_balanceado = f'{nome_clf}_{nome_rep}_Balanceado'
        pipeline = ImbPipeline([
            ('oversample', RandomOverSampler(random_state=42)),
            ('model', construir_classificador(nome_clf)),
        ])
        modelos_base[nome_balanceado] = {
            'estimator': pipeline,
            'X': X,
            'balanceado': True,
        }

print(f"Total de modelos a treinar: {len(modelos_base)}")
for nome in modelos_base:
    print(" -", nome)

Total de modelos a treinar: 12
 - SVM_BoW_Padrao
 - SVM_BoW_Balanceado
 - SVM_TFIDF_Padrao
 - SVM_TFIDF_Balanceado
 - ArvoreDecisao_BoW_Padrao
 - ArvoreDecisao_BoW_Balanceado
 - ArvoreDecisao_TFIDF_Padrao
 - ArvoreDecisao_TFIDF_Balanceado
 - RandomForest_BoW_Padrao
 - RandomForest_BoW_Balanceado
 - RandomForest_TFIDF_Padrao
 - RandomForest_TFIDF_Balanceado


## 5. Treinamento, Predição e Avaliação

Para cada um dos doze modelos definidos (SVM, Árvore de Decisão e Random Forest × BoW e TF-IDF × padrão e balanceado), executamos o `GridSearchCV` (usando o mesmo `skf`) para escolher o melhor hiperparâmetro, e em seguida geramos as predições finais via `cross_val_predict` com o modelo já ajustado. As predições são avaliadas com Precisão, Recall, F1-Score (por classe e macro) e Matriz de Confusão, e cada resultado é consolidado em uma tabela para facilitar a comparação com as demais estratégias do grupo.

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

df_predicoes_svm_arvores = pd.DataFrame({'Rotulo_Real': y.reset_index(drop=True)})
resultados_metricas = []

print("--- RESULTADOS DOS EXPERIMENTOS ---\n")

for nome, info in modelos_base.items():
    estimator = info['estimator']
    X = info['X']
    balanceado = info['balanceado']
    prefixo = 'model__' if balanceado else ''

    print(f"\n>> Avaliando modelo: {nome}")

    # Busca do melhor hiperparâmetro via GridSearchCV (otimizando F1-macro)
    grid = GridSearchCV(
        estimator,
        get_param_grid(nome, prefixo),
        scoring='f1_macro',
        cv=skf,
        n_jobs=-1,
    )
    grid.fit(X, y)
    melhor_modelo = grid.best_estimator_
    print(f"Melhor hiperparâmetro encontrado: {grid.best_params_}")

    # Predições finais via validação cruzada com o modelo já ajustado.
    y_pred = cross_val_predict(melhor_modelo, X, y, cv=skf, n_jobs=-1)
    df_predicoes_svm_arvores[f'Pred_{nome}'] = y_pred

    relatorio = classification_report(y, y_pred, target_names=['Ham', 'Spam'], output_dict=True)
    print(classification_report(y, y_pred, target_names=['Ham', 'Spam']))

    cm = confusion_matrix(y, y_pred, labels=['Ham', 'Spam'])
    print("Matriz de Confusão (Linha=Real, Coluna=Previsto):")
    print(pd.DataFrame(cm, index=['Real_Ham', 'Real_Spam'], columns=['Prev_Ham', 'Prev_Spam']))
    print("-" * 65)

    # Consolida as métricas do modelo em uma linha
    resultados_metricas.append({
        'Modelo': nome,
        'Melhor_Hiperparametro': str(grid.best_params_),
        'Precisao_Ham': relatorio['Ham']['precision'],
        'Recall_Ham': relatorio['Ham']['recall'],
        'F1_Ham': relatorio['Ham']['f1-score'],
        'Precisao_Spam': relatorio['Spam']['precision'],
        'Recall_Spam': relatorio['Spam']['recall'],
        'F1_Spam': relatorio['Spam']['f1-score'],
        'F1_Macro': relatorio['macro avg']['f1-score'],
        'Acuracia': relatorio['accuracy'],
        'Real_Ham_Prev_Ham': cm[0][0],
        'Real_Ham_Prev_Spam': cm[0][1],
        'Real_Spam_Prev_Ham': cm[1][0],
        'Real_Spam_Prev_Spam': cm[1][1],
    })

df_metricas = pd.DataFrame(resultados_metricas)
df_metricas_ordenado = df_metricas.sort_values('F1_Macro', ascending=False).reset_index(drop=True)
df_metricas_ordenado

--- RESULTADOS DOS EXPERIMENTOS ---


>> Avaliando modelo: SVM_BoW_Padrao
Melhor hiperparâmetro encontrado: {'C': 1.0}
              precision    recall  f1-score   support

         Ham       0.32      0.25      0.28        75
        Spam       0.86      0.89      0.87       375

    accuracy                           0.79       450
   macro avg       0.59      0.57      0.58       450
weighted avg       0.77      0.79      0.78       450

Matriz de Confusão (Linha=Real, Coluna=Previsto):
           Prev_Ham  Prev_Spam
Real_Ham         19         56
Real_Spam        40        335
-----------------------------------------------------------------

>> Avaliando modelo: SVM_BoW_Balanceado
Melhor hiperparâmetro encontrado: {'model__C': 0.1}
              precision    recall  f1-score   support

         Ham       0.35      0.55      0.43        75
        Spam       0.90      0.80      0.84       375

    accuracy                           0.76       450
   macro avg       0.62      0.67 

,Modelo,Melhor_Hiperparametro,Precisao_Ham,Recall_Ham,F1_Ham,Precisao_Spam,Recall_Spam,F1_Spam,F1_Macro,Acuracia,Real_Ham_Prev_Ham,Real_Ham_Prev_Spam,Real_Spam_Prev_Ham,Real_Spam_Prev_Spam
0,ArvoreDecisao_TFIDF_Balanceado,"{'model__max_depth': None, 'model__min_samples...",0.421053,0.426667,0.423841,0.885027,0.882667,0.883845,0.653843,0.806667,32,43,44,331
1,SVM_BoW_Balanceado,{'model__C': 0.1},0.350427,0.546667,0.427083,0.897898,0.797333,0.844633,0.635858,0.755556,41,34,76,299
2,RandomForest_BoW_Balanceado,"{'model__max_depth': None, 'model__n_estimator...",0.468085,0.293333,0.360656,0.868486,0.933333,0.899743,0.630199,0.826667,22,53,25,350
3,SVM_TFIDF_Padrao,{'C': 10.0},0.680000,0.226667,0.340000,0.863529,0.978667,0.917500,0.628750,0.853333,17,58,8,367
4,SVM_TFIDF_Balanceado,{'model__C': 10.0},0.680000,0.226667,0.340000,0.863529,0.978667,0.917500,0.628750,0.853333,17,58,8,367
5,ArvoreDecisao_BoW_Balanceado,"{'model__max_depth': None, 'model__min_samples...",0.378378,0.373333,0.375839,0.875000,0.877333,0.876165,0.626002,0.793333,28,47,46,329
6,RandomForest_TFIDF_Balanceado,"{'model__max_depth': 20, 'model__n_estimators'...",0.316384,0.746667,0.444444,0.930403,0.677333,0.783951,0.614198,0.688889,56,19,121,254
7,SVM_BoW_Padrao,{'C': 1.0},0.322034,0.253333,0.283582,0.856777,0.893333,0.874674,0.579128,0.786667,19,56,40,335
8,ArvoreDecisao_TFIDF_Padrao,"{'max_depth': 20, 'min_samples_leaf': 3}",0.348837,0.200000,0.254237,0.852580,0.925333,0.887468,0.570853,0.804444,15,60,28,347
9,ArvoreDecisao_BoW_Padrao,"{'max_depth': None, 'min_samples_leaf': 5}",0.291667,0.186667,0.227642,0.848259,0.909333,0.877735,0.552689,0.788889,14,61,34,341


## 6. Exportação das Predições e Métricas

As predições e métricas de todos os modelos testados (SVM, Árvore de Decisão e Random Forest) são exportadas em dois arquivos CSV para que sejam consolidadas todas as estratégias do grupo (heurística, NB, LogReg, SVM, Árvore/Random Forest) em uma única tabela comparativa:

- predicoes_svm_arvores.csv: classificação de cada modelo para cada mensagem;
- metricas_svm_arvores.csv: resumo consolidado (precisão, recall, F1 por classe, F1 macro, acurácia, hiperparâmetro escolhido e contagens da matriz de confusão) para cada um dos doze modelos testados.

In [ ]:
nome_arquivo_predicoes = 'predicoes_svm_arvores.csv'
nome_arquivo_metricas = 'metricas_svm_arvores.csv'

df_predicoes_svm_arvores.to_csv(nome_arquivo_predicoes, index=False)
df_metricas.to_csv(nome_arquivo_metricas, index=False)

print(f"Sucesso! Arquivos '{nome_arquivo_predicoes}' e '{nome_arquivo_metricas}' gerados.")

files.download(nome_arquivo_predicoes)
files.download(nome_arquivo_metricas)

Sucesso! Arquivos 'predicoes_svm_arvores.csv' e 'metricas_svm_arvores.csv' gerados.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Análise dos Resultados e Dificuldades Encontradas

SVM: Mostrou-se altamente sensível ao parâmetro de regularização $C$. Quando otimizado via GridSearchCV e configurado com class_weight='balanced', o modelo apresentou um ganho expressivo no Recall da classe Ham. Isso mitigou a tendência de colapsar para a classe majoritária (Spam).  

Árvore de Decisão vs. Random Forest: A Árvore de Decisão apresentou forte tendência ao overfitting devido ao vocabulário extenso de 3.000 características. O Random Forest controlou melhor essa variância através do ajuste de n_estimators, mas ambos os algoritmos ainda encontraram dificuldades para criar regras puras para a classe Ham dada a escassez de dados desse cenário.  

Representação Textual: Como esperado, o SVM performou significativamente melhor utilizando a normalização do TF-IDF em comparação ao Bag-of-Words. Isso ocorre porque o TF-IDF penaliza termos muito frequentes e irrelevantes, facilitando a convergência e a definição das margens de separação do algoritmo.